<a href="https://colab.research.google.com/github/mohammadayaanahsan/flyRank-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohammadayaanahsan/flyRank-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

The rule: a page is worth reviewing first if it is **stale** (not
updated in a while) but still **visible** (getting impressions),
**declining** while still in demand, **thin** content with real
traffic, sitting in a **page-one decay risk** position, or showing
**low CTR/engagement** despite visibility. Each page gets a
transparent weighted score, and any matching condition below adds a
reason code a human reviewer can read.

Reason codes:
- `stale_visible_page`: days_since_last_update >= 180 and impressions_90d >= 500
- `declining_with_demand`: trend_direction == "down" and impressions_90d >= 100
- `thin_visible_page`: 0 < word_count < 1200 and impressions_90d >= 250
- `page_one_decay_risk`: 0 < avg_position <= 10 and content_age_days >= 180
- `low_ctr_visible_page`: impressions_90d >= 500 and 0 < avg_position <= 20 and ctr < 0.5
- `low_engagement_visible_page`: sessions_90d >= 30 and (engagement_rate < 30 or scroll_rate < 30)

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 2. Build the ranked queue (writes the CSV)

Below, the baseline score is calculated for every page, reason codes
are attached, and the ranked queue is written to
`work/outputs/baseline_action_score.csv`.

In [2]:
!git clone https://github.com/mohammadayaanahsan/flyRank-internship.git
%cd flyRank-internship

Cloning into 'flyRank-internship'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 134 (delta 50), reused 95 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 1.85 MiB | 15.18 MiB/s, done.
Resolving deltas: 100% (50/50), done.
/content/flyRank-internship


In [3]:
# If fresh session, run this first:
# !git clone https://github.com/mohammadayaanahsan/flyRank-internship.git
# %cd flyRank-internship

import pandas as pd
import numpy as np
import os

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Normalize helper
def normalize(s):
    s = s.replace([np.inf, -np.inf], np.nan).fillna(0)
    if s.max() == s.min():
        return s * 0
    return (s - s.min()) / (s.max() - s.min())

visibility_score = normalize(df["impressions_90d"])
freshness_risk_score = normalize(df["days_since_last_update"])
position_opportunity_score = normalize(-df["avg_position"].fillna(df["avg_position"].max()))
depth_gap_score = normalize(-df["word_count"].fillna(df["word_count"].max()))

df["baseline_score"] = (
    0.40 * visibility_score
    + 0.30 * freshness_risk_score
    + 0.25 * position_opportunity_score
    + 0.05 * depth_gap_score
)

def reason_codes(row):
    codes = []
    if row.get("days_since_last_update", 0) >= 180 and row.get("impressions_90d", 0) >= 500:
        codes.append("stale_visible_page")
    if row.get("trend_direction") == "down" and row.get("impressions_90d", 0) >= 100:
        codes.append("declining_with_demand")
    if 0 < row.get("word_count", 0) < 1200 and row.get("impressions_90d", 0) >= 250:
        codes.append("thin_visible_page")
    if 0 < row.get("avg_position", 0) <= 10 and row.get("content_age_days", 0) >= 180:
        codes.append("page_one_decay_risk")
    if row.get("impressions_90d", 0) >= 500 and 0 < row.get("avg_position", 0) <= 20 and row.get("ctr", 1) < 0.5:
        codes.append("low_ctr_visible_page")
    if row.get("sessions_90d", 0) >= 30 and (row.get("engagement_rate", 100) < 30 or row.get("scroll_rate", 100) < 30):
        codes.append("low_engagement_visible_page")
    return ",".join(codes) if codes else "none"

df["reason_codes"] = df.apply(reason_codes, axis=1)

ranked = df.sort_values("baseline_score", ascending=False)

os.makedirs("work/outputs", exist_ok=True)
ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Ranked {len(ranked)} pages. Saved to work/outputs/baseline_action_score.csv")
print(ranked[["content_id", "baseline_score", "reason_codes"]].head(10))

Ranked 30000 pages. Saved to work/outputs/baseline_action_score.csv
                 content_id  baseline_score  \
6653   content_5fe46e04994d        0.728779   
26844  content_8c19996aa890        0.691099   
19636  content_2cb567c3c89b        0.667436   
17812  content_aaef01a50def        0.660957   
29400  content_2dba2b1f9536        0.657006   
21819  content_4c36c775b818        0.654588   
26531  content_cb112fce36be        0.602363   
13537  content_2c2606c5d176        0.597188   
29879  content_1a9e894be2e2        0.584405   
21565  content_9532f197bbc8        0.569913   

                                            reason_codes  
6653   declining_with_demand,page_one_decay_risk,low_...  
26844  declining_with_demand,page_one_decay_risk,low_...  
19636                        low_engagement_visible_page  
17812  page_one_decay_risk,low_ctr_visible_page,low_e...  
29400                        low_engagement_visible_page  
21819  declining_with_demand,page_one_decay_risk,low_...  
2

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 3. Top-20 review

Below, the top 20 ranked pages are inspected directly: their score,
reason codes, and a manual confidence note.

In [4]:
top20 = ranked.head(20)[["content_id", "baseline_score", "reason_codes",
                          "impressions_90d", "days_since_last_update",
                          "avg_position", "trend_direction"]]
print(top20.to_string(index=False))

          content_id  baseline_score                                                                               reason_codes  impressions_90d  days_since_last_update  avg_position trend_direction
content_5fe46e04994d        0.728779 declining_with_demand,page_one_decay_risk,low_ctr_visible_page,low_engagement_visible_page           517715                     104           4.2            down
content_8c19996aa890        0.691099 declining_with_demand,page_one_decay_risk,low_ctr_visible_page,low_engagement_visible_page           509252                      20           2.5            down
content_2cb567c3c89b        0.667436                                                                low_engagement_visible_page           497727                      48          22.2              up
content_aaef01a50def        0.660957                       page_one_decay_risk,low_ctr_visible_page,low_engagement_visible_page           517109                      22           5.4          stable
conte

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 4. Weak picks + leakage check

Below, potential weak picks are flagged (single reason code, borderline
volume) and a leakage check confirms no product-decision fields or
future-window metrics were used as inputs.

In [5]:
weak_picks = ranked.head(20)[ranked.head(20)["reason_codes"].apply(lambda x: x.count(",") == 0)]
print(f"Top-20 pages with only ONE reason code (weaker confidence): {len(weak_picks)}")
print(weak_picks[["content_id", "reason_codes", "baseline_score"]])

print("\nLeakage check — columns used in scoring:")
print(["impressions_90d", "days_since_last_update", "avg_position", "word_count",
       "trend_direction", "ctr", "sessions_90d", "engagement_rate", "scroll_rate"])
print("None of these are product decision flags (health_score, priority_score, "
      "action_type) — all are observable signals available before any decision was made.")

Top-20 pages with only ONE reason code (weaker confidence): 10
                 content_id                 reason_codes  baseline_score
19636  content_2cb567c3c89b  low_engagement_visible_page        0.667436
29400  content_2dba2b1f9536  low_engagement_visible_page        0.657006
6962   content_f01216059a6a          page_one_decay_risk        0.557334
15608  content_06e19c6486b0          page_one_decay_risk        0.556680
8631   content_e2b702f4f92b          page_one_decay_risk        0.552591
4606   content_3f3576c295f5          page_one_decay_risk        0.548980
26242  content_55a5b1c46474          page_one_decay_risk        0.542373
24216  content_1b4ec72dafd4          page_one_decay_risk        0.542051
26798  content_b28d1efd668f  low_engagement_visible_page        0.541636
14090  content_44e481c8f55b  low_engagement_visible_page        0.539595

Leakage check — columns used in scoring:
['impressions_90d', 'days_since_last_update', 'avg_position', 'word_count', 'trend_direction

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.